# 📘 Tutorial 5: Vectorized Calculations and Derived Columns

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

Once numeric columns are clean, the next step is often to calculate new columns: unit conversions, baseline-subtracted signals, normalized responses, or rates. pandas and NumPy make these operations clearer when we write them as vectorized column operations.

---

## 🎯 Learning objectives

By the end of this notebook, you should be able to:

- use vectorized operations on columns,
- convert units,
- subtract a baseline,
- normalize a response,
- create derived columns,
- preserve original columns where useful.


## 🔧 Setup


In [ ]:
import pandas as pd


## 1. Start from clean numeric data


In [ ]:
measurements = pd.DataFrame(
    {
        "sample_id": ["S01", "S02", "S03", "S04"],
        "condition": ["control", "low", "medium", "high"],
        "concentration_uM": [0, 50, 100, 200],
        "blank_absorbance": [0.045, 0.045, 0.045, 0.045],
        "absorbance": [0.052, 0.214, 0.392, 0.735],
    }
)

measurements


The table is small, but it has several common ingredients: units, a blank measurement, and a response that might need normalization.


## 2. Unit conversion


In [ ]:
measurements["concentration_mM"] = measurements["concentration_uM"] / 1000
measurements


The operation happens for the whole column at once. There is no need to loop over rows.


## 3. Baseline subtraction


In [ ]:
measurements["net_absorbance"] = (
    measurements["absorbance"] - measurements["blank_absorbance"]
)

measurements[["sample_id", "absorbance", "blank_absorbance", "net_absorbance"]]


The original columns remain available. That makes the derived column auditable.


## 4. Normalization


In [ ]:
max_net_absorbance = measurements["net_absorbance"].max()
measurements["relative_response"] = measurements["net_absorbance"] / max_net_absorbance

measurements[["sample_id", "net_absorbance", "relative_response"]]


Normalization changes the scale but not the ordering. It is useful when later plots need responses on a comparable scale.


## 5. Vectorized conditions


In [ ]:
measurements["is_above_half_max"] = measurements["relative_response"] >= 0.5
measurements[["sample_id", "condition", "relative_response", "is_above_half_max"]]


Boolean columns can be useful for filtering, grouping, and annotating later plots.


## 6. Why not a manual loop?


In [ ]:
loop_result = []
for value in measurements["absorbance"]:
    loop_result.append(value - 0.045)

loop_result


The loop works, but it hides the column relationship and duplicates information already present in the table. The vectorized version is shorter and tied directly to column names.


In [ ]:
measurements["net_absorbance_vectorized"] = (
    measurements["absorbance"] - measurements["blank_absorbance"]
)

measurements[["net_absorbance", "net_absorbance_vectorized"]]


## 7. Build a plot-ready table


In [ ]:
plot_ready = measurements[
    ["condition", "concentration_mM", "net_absorbance", "relative_response"]
].copy()

plot_ready


In [ ]:
x = plot_ready["concentration_mM"].to_numpy()
y = plot_ready["relative_response"].to_numpy()

print("x:", x)
print("y:", y)


This table now has clean columns that a later plotting notebook could use directly.


## 8. Checkpoint: add a percentage column

Task: convert `relative_response` into a percentage while preserving the original relative column.


In [ ]:
plot_ready["relative_response_percent"] = plot_ready["relative_response"] * 100
plot_ready


## 🧭 Key takeaways

- Vectorized column operations are usually clearer than manual loops.
- Derived columns should have names that describe the calculation.
- Preserve original columns when the derived value should be auditable.
- Unit conversion, baseline subtraction, and normalization are common pre-plotting steps.
- The goal is a table with explicit x, y, and grouping columns.
